In [ ]:
import pandas as pd
import os

# ---------------------------------------------------------
# CONFIGURACIÓN DE ARCHIVOS
# ---------------------------------------------------------
input_file_path = "../data/raw/tariff_database_2026.xlsx"
output_file_path = "../data/intermediate/fracciones_tmec.xlsx"

# ---------------------------------------------------------
# LÓGICA DE PROCESAMIENTO
# ---------------------------------------------------------

def procesar_tarifas():
    if not os.path.exists(input_file_path):
        print(f"Error: No se encuentra el archivo en: {input_file_path}")
        return

    print("Cargando archivo de Excel...")
    
    try:
        # Leemos como string para no perder ceros a la izquierda en códigos
        df = pd.read_excel(input_file_path, dtype=str)
    except Exception as e:
        print(f"Error al leer el archivo Excel: {e}")
        return

    # Selección de columnas
    columnas_interes = [
        "hts8", "quantity_1_code", "quantity_2_code", "col1_special_text", 
        "usmca_indicator", "usmca_rate_type_code", "usmca_ad_val_rate", 
        "usmca_specific_rate", "usmca_other_rate"
    ]
    
    # Validamos existencia de columnas
    cols_existentes = [c for c in columnas_interes if c in df.columns]
    df = df[cols_existentes]

    # Filtrar indicadores S o S+
    df['usmca_indicator'] = df['usmca_indicator'].str.strip().str.upper()
    df_filtrado = df[df['usmca_indicator'].isin(['S', 'S+'])].copy()

    print(f"Registros filtrados (S o S+): {len(df_filtrado)}")

    # -----------------------------------------------------
    # FUNCIONES DE FORMATEO Y LIMPIEZA
    # -----------------------------------------------------

    def es_tasa_compleja(valor_str):
        """Detecta si el valor contiene el marcador 9999.99"""
        if "9999.99" in valor_str:
            return True
        return False

    def formatear_moneda(valor):
        """
        Convierte un valor string (ej '0.5') a formato moneda ('$0.5').
        Si es vacío, devuelve '0'.
        """
        if pd.isna(valor) or str(valor).strip() == "":
            return "$0"
        
        s_val = str(valor).strip()
        if es_tasa_compleja(s_val):
            return "COMPLEX"
        
        try:
            # Intentamos convertir a float para quitar ceros innecesarios si se desea,
            # o simplemente concatenamos el $. Aquí concatenamos directo para ser fieles al dato.
            # Si prefieres quitar ceros a la derecha, usa float.
            return f"${s_val}"
        except:
            return f"${s_val}"

    def formatear_porcentaje(valor):
        """
        Convierte un valor string decimal (ej '0.05') a porcentaje ('5%').
        """
        if pd.isna(valor) or str(valor).strip() == "":
            return "0%"
        
        s_val = str(valor).strip()
        if es_tasa_compleja(s_val):
            return "COMPLEX"
        
        try:
            # Convertir a float y multiplicar por 100
            float_val = float(s_val)
            porcentaje = float_val * 100
            # Usamos :g para evitar decimales innecesarios (5.0 -> 5)
            return f"{porcentaje:g}%"
        except:
            # Si falla la conversión (ej. texto raro), devolvemos tal cual
            return f"{s_val}"

    def construir_parte_especifica(tasa_formateada, unidad_codigo):
        """
        Maneja la lógica: Tasa * Unidad.
        Si Unidad está vacía -> Tasa/unidad.
        """
        if tasa_formateada == "COMPLEX":
            return "COMPLEX"
        
        unidad = str(unidad_codigo).strip() if not pd.isna(unidad_codigo) else ""
        
        if unidad == "":
            return f"{tasa_formateada}/unidad"
        else:
            return f"{tasa_formateada} * {unidad}"

    # -----------------------------------------------------
    # FUNCIÓN PRINCIPAL DE CÁLCULO
    # -----------------------------------------------------

    def calcular_final_rate(row):
        code = str(row.get('usmca_rate_type_code', '')).strip()
        
        # Obtenemos valores crudos
        raw_spec = row.get('usmca_specific_rate')
        raw_other = row.get('usmca_other_rate')
        raw_adval = row.get('usmca_ad_val_rate')
        
        q1 = row.get('quantity_1_code')
        q2 = row.get('quantity_2_code')

        # Formateamos valores
        spec_fmt = formatear_moneda(raw_spec)      # Ej: $0.4
        other_fmt = formatear_moneda(raw_other)    # Ej: $0.1
        adval_fmt = formatear_porcentaje(raw_adval)# Ej: 5%

        # Verificamos si alguno resultó ser complejo antes de armar la fórmula
        if "COMPLEX" in [spec_fmt, other_fmt, adval_fmt]:
            return "Tasa Compleja (Revisar Aquí)"

        # Construimos los bloques lógicos (Ej: "$0.4 * KG" o "$0.4/unidad")
        bloque_spec_q1 = construir_parte_especifica(spec_fmt, q1)
        bloque_spec_q2 = construir_parte_especifica(spec_fmt, q2)
        bloque_other_q2 = construir_parte_especifica(other_fmt, q2)
        
        # Bloque Ad Valorem sin " * Valor"
        bloque_adval = adval_fmt

        # Aplicamos reglas según el código (sin paréntesis)
        if code == '0':
            return "Free"
        
        elif code == '1':
            return bloque_spec_q1
        
        elif code == '2':
            return bloque_spec_q2
        
        elif code == '3':
            return f"{bloque_spec_q1} + {bloque_other_q2}"
        
        elif code == '4':
            return f"{bloque_spec_q1} + {bloque_adval}"
        
        elif code == '5':
            return f"{bloque_spec_q2} + {bloque_adval}"
        
        elif code == '6':
            return f"{bloque_spec_q1} + {bloque_other_q2} + {bloque_adval}"
        
        elif code == '7':
            return bloque_adval
        
        elif code == '9':
            return f"{adval_fmt} * Derived Duty (Refer to HTS)"
        
        elif code in ['K', 'X']:
            return "Refer to HTS for duty computation procedures"
        
        elif code == 'T':
            return "Compute at 10-digit level. Refer to HTS"
        
        else:
            # Si el código viene vacío o extraño
            return f"Unknown Code ({code})"

    print("Calculando 'usmca_final_rate' con nuevas reglas de formato...")
    df_filtrado['usmca_final_rate'] = df_filtrado.apply(calcular_final_rate, axis=1)

    print(f"Guardando archivo final: {output_file_path}")
    df_filtrado.to_excel(output_file_path, index=False)
    print("¡Proceso finalizado!")

if __name__ == "__main__":
    procesar_tarifas()

Cargando archivo de Excel...
Registros filtrados (S o S+): 7289
Calculando 'usmca_final_rate' con nuevas reglas de formato...
Guardando archivo final: ../data/intermediate/fracciones_tmec.xlsx
¡Proceso finalizado!
